Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Agent state

 **The shared contract between every node in the graph.**

 Every node receives the full `AgentState` and returns a partial dict.
 LangGraph merges the partial update into the state before passing it to the next node.
 No node needs to know about other nodes — they just read what they need
 and write what they produce.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

DEFAULT_MAX_ITERATIONS: int = 10

 ## `add_messages` reducer

 The only field with a reducer. Every other field is last-write-wins.
 `add_messages` appends instead of overwrites — every node that returns
 `{"messages": [new_msg]}` adds to the list, never replaces it.

 Without this annotation, nodes would overwrite each other's messages
 and the agent would lose its conversation history mid-run.

 ## Field ownership

 | Field | Set by |
 |---|---|
 | `messages` | every node |
 | `current_plan`, `selected_tools` | planner |
 | `input_validated`, `risk_level`, `blocked` | input_guard |
 | `iterations`, `max_iterations` | input_guard (init) · agent_llm (increment) |
 | `requires_human_approval`, `human_approved` | planner + input_guard (set) · human_loop (resolve) |
 | `output_validated`, `final_response` | output_guard |
 | `workflow_trace` | every node (appends) |
 | `user_context` | loaded from LTM at session start |

 ## Important

 `TypedDict` fields have no defaults — reading a field before any node has written it
 raises a `KeyError`. Always use `state.get("field", default)` across all nodes.

In [ ]:
class AgentState(TypedDict):

    # Conversation
    messages: Annotated[list[BaseMessage], add_messages]

    # Planning
    current_plan: list[str]
    current_step: int
    task_complete: bool

    # Tools
    selected_tools: list[str]
    tool_call_history: list[dict]

    # Memory
    user_context: dict
    session_id: str
    user_id: str

    # Guardrails
    input_validated: bool
    output_validated: bool
    iterations: int
    max_iterations: int
    requires_human_approval: bool
    human_approved: bool | None

    # UI flow
    workflow_trace: list[dict]      # [{node, status, timestamp}]
    intermediate_results: list[dict]
    final_response: str | None

    # Security
    risk_level: Literal["low", "medium", "high"]
    blocked: bool
    block_reason: str | None
    hallucination_warning: bool     # surfaced by output_guard